# Urna Aberta: independently verifiable synthetic election

This notebook reads only public files under `demo/results/`. The offline verifier checks the signed batches, receipts, board evidence, and public trustee partials in a separate process. Copied evidence is not consensus or finality proof.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

results = Path('demo/results')
manifest_path = results / 'manifest.json'
evidence_path = results / 'evidence.json'
receipts_path = results / 'receipts.json'
for path in (manifest_path, evidence_path, receipts_path):
    if not path.is_file():
        raise FileNotFoundError(f'Required public capture is missing: {path.as_posix()}')

command = [sys.executable, '-m', 'verifier.cli', '--manifest', str(manifest_path), '--evidence', str(evidence_path), '--receipts', str(receipts_path)]
verified = subprocess.run(command, check=True, capture_output=True, text=True)
report = json.loads(verified.stdout)
report

{'assumptions': ['offline copied JSON does not prove consensus or finality',
  'complete honest board export is assumed',
  'CAST ballots are counted; SPOILED ballots are excluded'],
 'report': {'aggregate_digest': 'da6aca85f84e89a17f6ccb8b7e54fbdfe1b1552469f480e1ac68c9670182b038',
  'cast_count': 6,
  'election_id': '91e9ca6ae7350373c6744193bc5503989d6fc7848d507a8a4c4c7e86ae46bde5',
  'manifest_hash': '71c8a56acf5118a5106b9fdd38e3e7f8a2af13ade4fc7d06d9550494513cb45f',
  'receipt_results': {'01ac66eb54cb4b7ebc7fae42ddb6203b8ce78ac03aa60b6cfc8358825b017008': 'CAST',
   '07ccda3c88bf07e2eb282c5616d55acf34df4ad85789ef327222dbf879a2f758': 'CAST',
   '6335ca677b506d30daf37709b746e2bd367210ffb4055777f4959eb69bc9ab11': 'CAST',
   '686bb4ed790860b1883890eee9f7bc761cd4d4e3ef684e950c4e613c1bbc7e0d': 'SPOILED',
   '7b6319dce96d78492584a6e9d296410f07bac49b75130028453abcb0b8cfd1d7': 'CAST',
   '84be236f8ec7ee93d4d15e421dca0fdef93c7fb14c4f5b51f5249112020142e5': 'CAST',
   '89fd579745bbaeba20c41ca69a

In [2]:
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
measurements_path = results / 'measurements.json'
measurements = json.loads(measurements_path.read_text(encoding='utf-8')) if measurements_path.is_file() else None
observations = {
    'candidates': manifest['candidates'],
    'registered_machines': len(manifest['machines']),
    'published_batches': len(evidence['batches']),
    'posted_trustee_partials': len(evidence['partials']),
    'CAST count': report['report']['cast_count'],
    'SPOILED count (excluded)': report['report']['spoiled_count'],
    'candidate totals': report['report']['totals'],
    'measurement capture present': measurements is not None,
}
observations

{'candidates': ['Ada Lovelace', 'Grace Hopper', 'Katherine Johnson'],
 'registered_machines': 2,
 'published_batches': 2,
 'posted_trustee_partials': 3,
 'CAST count': 6,
 'SPOILED count (excluded)': 1,
 'candidate totals': [2, 2, 2],
 'measurement capture present': True}

In [3]:
if measurements is None:
    raise FileNotFoundError('Required measured observations are missing')
print(json.dumps(measurements['raw_samples'], indent=2))

{
  "full_demo": [
    {
      "cache_note": "first process/node is cold; subsequent fresh processes/nodes follow cache warming",
      "classification": "cold",
      "fresh_node": true,
      "fresh_process": true,
      "monotonic_ns": 56590681917,
      "sample": 1
    },
    {
      "cache_note": "first process/node is cold; subsequent fresh processes/nodes follow cache warming",
      "classification": "warm",
      "fresh_node": true,
      "fresh_process": true,
      "monotonic_ns": 56174527333,
      "sample": 2
    },
    {
      "cache_note": "first process/node is cold; subsequent fresh processes/nodes follow cache warming",
      "classification": "warm",
      "fresh_node": true,
      "fresh_process": true,
      "monotonic_ns": 56164920292,
      "sample": 3
    }
  ],
  "pure_operations": [
    {
      "metric": "encryption_full_candidate_vector",
      "monotonic_ns": 151508750,
      "sample": 1
    },
    {
      "metric": "encryption_full_candidate_vector",
      